In [120]:
import numpy as np
import pandas as pd

#Geolocalização
import reverse_geocoder as rg
import pycountry_convert as pc
from global_land_mask import globe

# Higienização com Dados Nulos

In [121]:
# Carregar o dataset

df = pd.read_csv('Dataset/alert_historico_sismos_api.csv')
#df = pd.read_csv('Dataset/earthquakes.csv')

# Exibir as primeiras linhas originais e o tamanho do dataset
print(f"Tamanho original do dataset: {df.shape}")
display(df.head())

Tamanho original do dataset: (33710, 29)


,mag,place,time,updated,tz,url,detail,felt,cdi,mmi,...,nst,dmin,rms,gap,magType,type,title,longitude,latitude,depth
0,5.1,"134 km WSW of Bengkulu, Indonesia",2008-12-30 20:32:38.020,1415324284737,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,71.0,NaN,1.26,103.9,mb,earthquake,"M 5.1 - 134 km WSW of Bengkulu, Indonesia",101.191,-4.366,10.0
1,5.9,"128 km WSW of Bengkulu, Indonesia",2008-12-30 19:49:52.610,1651521482332,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,214.0,NaN,1.10,27.7,mwb,earthquake,"M 5.9 - 128 km WSW of Bengkulu, Indonesia",101.217,-4.297,20.0
2,5.2,"189 km S of Dompu, Indonesia",2008-12-30 18:09:24.770,1415324284682,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,78.0,NaN,1.00,87.8,mb,earthquake,"M 5.2 - 189 km S of Dompu, Indonesia",118.459,-10.248,49.3
3,5.4,"84 km WSW of Kirakira, Solomon Islands",2008-12-30 17:43:51.200,1651521480662,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,102.0,NaN,0.93,53.4,mwc,earthquake,"M 5.4 - 84 km WSW of Kirakira, Solomon Islands",161.162,-10.619,57.7
4,5.1,"27 km E of Sarangani, Philippines",2008-12-30 03:20:56.570,1766417714439,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,123.0,NaN,0.98,48.9,mwc,earthquake,"M 5.1 - 27 km E of Sarangani, Philippines",125.710,5.419,141.3


### Encontrando `continent`, `country` e `subnational` com reverse_geocoder

In [122]:
print("Classificando Terra vs Oceano...")

# 1. A global_land_mask verifica instantaneamente todas as latitudes e longitudes
# Retorna uma lista de Verdadeiros (Terra) e Falsos (Oceano)
is_land = globe.is_land(df['latitude'].values, df['longitude'].values)

# 2. Inicializamos todos mundo como Oceano por padrão
df['country'] = 'Ocean/Offshore'
df['subnational'] = 'Unknown'

# 3. Filtramos APENAS os terremotos que aconteceram na terra
df_terra = df[is_land]
coordenadas_terra = list(zip(df_terra['latitude'], df_terra['longitude']))

print(f"Encontrados {len(coordenadas_terra)} sismos na terra. Buscando países...")

# 4. Rodamos o reverse_geocoder só para os sismos terrestres
resultados_terra = rg.search(coordenadas_terra)

paises_codigo = [res['cc'] for res in resultados_terra]
subnacionais = [res.get('admin1', 'Unknown') for res in resultados_terra]

def codigo_para_nome(codigo):
    try:
        return pc.country_alpha2_to_country_name(codigo)
    except:
        return 'Unknown'

paises_nome = [codigo_para_nome(c) for c in paises_codigo]

# 5. Inserimos os países e estados descobertos de volta no DataFrame original
# Usamos o .loc para atualizar apenas as linhas onde 'is_land' é True
df.loc[is_land, 'country'] = paises_nome
df.loc[is_land, 'subnational'] = subnacionais

print("\nConcluído!")

Classificando Terra vs Oceano...
Encontrados 6635 sismos na terra. Buscando países...

Concluído!


In [123]:
df["country"].value_counts()

country
Ocean/Offshore          27075
Indonesia                 766
Papua New Guinea          642
China                     605
Chile                     572
                        ...  
Guyana                      1
Jamaica                     1
Slovakia                    1
Syrian Arab Republic        1
Greenland                   1
Name: count, Length: 109, dtype: int64

### Renomeação e Remoção de Colunas

In [124]:
## Lista de colunas para remover baseada na análise prévia
#colunas_para_remover = [
#    'id', 'code', 'url', 'detailUrl', 'what3words', 'ids', # Identificadores e Links
#    'type', 'geometryType',                                # Valores constantes
#    'updated', 'time',                                     # Metadados de sistema/timestamps duplicados
#    'title', 'locationDetails', 'place', 'placeOnly',      # Textos redundantes ou complexos
#    'location', 'city', 'locality',                        # Textos geográficos propensos a erros
#    'postcode'                                             # Excesso de dados nulos
#]

## Removendo as colunas
#df_clean = df.drop(columns=colunas_para_remover)


# 1. Renomear colunas para manter o padrão que seus modelos já conhecem
df = df.rename(columns={'mag': 'magnitude', 'time': 'date'})

# 2. Extrair a distância em KM do texto da coluna 'place' (ex: "58 km S of Tokyo")
df['distanceKM'] = df['place'].str.extract(r'^(\d+)\s*km').astype(float)
# Preencher valores onde a distância não foi informada com 0
df['distanceKM'] = df['distanceKM'].fillna(0)

# 3. Remover as colunas inúteis específicas da API
colunas_para_remover = ['url', 'detail', 'tz', 'status', 'net', 'code', 'ids', 'sources', 'types', 'title', 'place', 'type']
df_clean = df.drop(columns=colunas_para_remover, errors='ignore')

# Eliminar colunas que causam vazamento de dados ou não estão disponíveis no tempo real
df_clean = df_clean.drop(columns=['felt', 'cdi', 'mmi'])

# 4. Tratar os nulos do Alvo (O que não tem cor, não teve alerta)
df_clean['alert'] = df_clean['alert'].fillna('No_Alert')

print(f"Tamanho após remoção de colunas: {df_clean.shape}")

Tamanho após remoção de colunas: (33710, 17)


### Transformação das Datas

In [125]:
# Converter a coluna 'date' para o formato datetime
df_clean['date'] = pd.to_datetime(df_clean['date'])

# (Opcional) Extrair ano, mês e hora para ajudar modelos de Machine Learning
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['day'] = df_clean['date'].dt.day
#df_clean['hour'] = df_clean['date'].dt.hour

# Exibir os tipos de dados atualizados
print(df_clean.dtypes)

magnitude             float64
date           datetime64[us]
updated                 int64
alert                     str
tsunami                 int64
sig                     int64
nst                   float64
dmin                  float64
rms                   float64
gap                   float64
magType                   str
longitude             float64
latitude              float64
depth                 float64
country                   str
subnational               str
distanceKM            float64
year                    int32
month                   int32
day                     int32
dtype: object


### (OPCIONAL)Preenchimento dos Valores Nulos em `nst`, `dmin`, `rms` e `gap`

Os modelos baseados em árvore conseguem lidar bem com os valores nulos

In [126]:
# colunas_sensores = ['nst', 'dmin', 'rms', 'gap']
# for col in colunas_sensores:
#     mediana = df_clean[col].median()
#     df_clean[col] = df_clean[col].fillna(mediana)

### Valores Nulos Gerais

In [127]:
# Verificando a quantidade de nulos antes do tratamento
print("Valores nulos antes do tratamento:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print(f"Tamanho antes do tratamento de nulos: {df_clean.shape}")

# # Preenchendo valores nulos em colunas categóricas com 'Unknown'
# colunas_categoricas_com_nulos = ['alert']#, 'continent', 'country', 'subnational']
# for col in colunas_categoricas_com_nulos:
#     if col in df_clean.columns:
#         df_clean[col] = df_clean[col].fillna('Unknown')


Valores nulos antes do tratamento:
nst     14981
dmin    12202
rms       799
gap       573
dtype: int64
Tamanho antes do tratamento de nulos: (33710, 20)


In [128]:
# Se houver alguma outra linha que ainda tenha nulos (dados corrompidos), nós a removemos
# 1. Faça uma lista com o nome das colunas que você quer IGNORAR na hora de apagar as linhas
colunas_ignoradas = ['nst', 'dmin', 'rms', 'gap']

# 2. Pegue todas as colunas do dataset, EXCETO as que estão na lista acima
colunas_para_verificar = df_clean.columns.difference(colunas_ignoradas)

# 3. Aplique o dropna usando apenas as colunas filtradas no 'subset'
df_clean = df_clean.dropna(subset=colunas_para_verificar)

In [129]:
print("\nValores nulos após o tratamento:")
# print(df_clean.isnull().sum().max()) # Deve retornar 0
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print(f"Tamanho após tratamento de nulos: {df_clean.shape}")


Valores nulos após o tratamento:
nst     14981
dmin    12202
rms       799
gap       573
dtype: int64
Tamanho após tratamento de nulos: (33710, 20)


### Linhas Duplicadas

In [130]:
# 1. Verificar quantas linhas estão exatamente duplicadas
quantidade_duplicados = df_clean.duplicated().sum()
print(f"Linhas duplicadas encontradas antes: {quantidade_duplicados}")

Linhas duplicadas encontradas antes: 0


In [131]:
df_clean = df_clean.drop_duplicates()

In [132]:
quantidade_duplicados = df_clean.duplicated().sum()
print(f"Linhas duplicadas encontradas depois: {quantidade_duplicados}")

Linhas duplicadas encontradas antes: 0


# Salvando o Dataset

In [133]:
# Visualizar o dataset higienizado
print(df_clean.head())

# Salvar o novo dataset limpo
arquivo_limpo = 'Dataset/alert_earthquakes_cleaned_with_null.csv'
df_clean.to_csv(arquivo_limpo, index=False)

print(f"\nHigienização concluída! O dataset limpo foi salvo como: {arquivo_limpo}")

   magnitude                    date        updated     alert  tsunami  sig  \
0        5.1 2008-12-30 20:32:38.020  1415324284737  No_Alert        0  400   
1        5.9 2008-12-30 19:49:52.610  1651521482332  No_Alert        0  536   
2        5.2 2008-12-30 18:09:24.770  1415324284682  No_Alert        0  416   
3        5.4 2008-12-30 17:43:51.200  1651521480662  No_Alert        0  449   
4        5.1 2008-12-30 03:20:56.570  1766417714439  No_Alert        0  400   

     nst  dmin   rms    gap magType  longitude  latitude  depth  \
0   71.0   NaN  1.26  103.9      mb    101.191    -4.366   10.0   
1  214.0   NaN  1.10   27.7     mwb    101.217    -4.297   20.0   
2   78.0   NaN  1.00   87.8      mb    118.459   -10.248   49.3   
3  102.0   NaN  0.93   53.4     mwc    161.162   -10.619   57.7   
4  123.0   NaN  0.98   48.9     mwc    125.710     5.419  141.3   

          country subnational  distanceKM  year  month  day  
0  Ocean/Offshore     Unknown       134.0  2008     12   30 

# Higienização sem Dados Nulos

In [ ]:
import numpy as np
import pandas as pd

# Para o uso do Geopy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

# Higienização
# Carregar o dataset
df = pd.read_csv('Dataset/alert_historico_sismos_api.csv')

# Exibir as primeiras linhas originais e o tamanho do dataset
print(f"Tamanho original do dataset: {df.shape}")
display(df.head())
### Renomeação e Remoção de Colunas
## Lista de colunas para remover baseada na análise prévia
#colunas_para_remover = [
#    'id', 'code', 'url', 'detailUrl', 'what3words', 'ids', # Identificadores e Links
#    'type', 'geometryType',                                # Valores constantes
#    'updated', 'time',                                     # Metadados de sistema/timestamps duplicados
#    'title', 'locationDetails', 'place', 'placeOnly',      # Textos redundantes ou complexos
#    'location', 'city', 'locality',                        # Textos geográficos propensos a erros
#    'postcode'                                             # Excesso de dados nulos
#]

## Removendo as colunas
#df_clean = df.drop(columns=colunas_para_remover)


# 1. Renomear colunas para manter o padrão que seus modelos já conhecem
df = df.rename(columns={'mag': 'magnitude', 'time': 'date'})

# 2. Extrair a distância em KM do texto da coluna 'place' (ex: "58 km S of Tokyo")
df['distanceKM'] = df['place'].str.extract(r'^(\d+)\s*km').astype(float)
# Preencher valores onde a distância não foi informada com 0
df['distanceKM'] = df['distanceKM'].fillna(0)

# 3. Remover as colunas inúteis específicas da API
colunas_para_remover = ['url', 'detail', 'tz', 'status', 'net', 'code', 'ids', 'sources', 'types', 'title', 'place',
                        'type']
df_clean = df.drop(columns=colunas_para_remover, errors='ignore')

# Eliminar colunas que causam vazamento de dados ou não estão disponíveis no tempo real
df_clean = df_clean.drop(columns=['felt', 'cdi', 'mmi'])

# 4. Tratar os nulos do Alvo (O que não tem cor, não teve alerta)
df_clean['alert'] = df_clean['alert'].fillna('No_Alert')

print(f"Tamanho após remoção de colunas: {df_clean.shape}")
### Transformação das Datas
# Converter a coluna 'date' para o formato datetime
df_clean['date'] = pd.to_datetime(df_clean['date'])

# (Opcional) Extrair ano, mês e hora para ajudar modelos de Machine Learning
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['day'] = df_clean['date'].dt.day
#df_clean['hour'] = df_clean['date'].dt.hour

# Exibir os tipos de dados atualizados
print(df_clean.dtypes)
### (OPCIONAL)Preenchimento dos Valores Nulos em `nst`, `dmin`, `rms` e `gap`

Os
modelos
baseados
em
árvore
conseguem
lidar
bem
com
os
valores
nulos
# colunas_sensores = ['nst', 'dmin', 'rms', 'gap']
# for col in colunas_sensores:
#     mediana = df_clean[col].median()
#     df_clean[col] = df_clean[col].fillna(mediana)
### Valores Nulos Gerais
# Verificando a quantidade de nulos antes do tratamento
print("Valores nulos antes do tratamento:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print(f"Tamanho antes do tratamento de nulos: {df_clean.shape}")

# # Preenchendo valores nulos em colunas categóricas com 'Unknown'
# colunas_categoricas_com_nulos = ['alert']#, 'continent', 'country', 'subnational']
# for col in colunas_categoricas_com_nulos:
#     if col in df_clean.columns:
#         df_clean[col] = df_clean[col].fillna('Unknown')

# Se houver alguma outra linha que ainda tenha nulos (dados corrompidos), nós a removemos
# 1. Faça uma lista com o nome das colunas que você quer IGNORAR na hora de apagar as linhas
colunas_ignoradas = ['nst', 'dmin', 'rms', 'gap']

# 2. Pegue todas as colunas do dataset, EXCETO as que estão na lista acima
colunas_para_verificar = df_clean.columns.difference(colunas_ignoradas)

# 3. Aplique o dropna usando apenas as colunas filtradas no 'subset'
df_clean = df_clean.dropna(subset=colunas_para_verificar)

print("\nValores nulos após o tratamento:")
print(df_clean.isnull().sum().max())  # Deve retornar 0
print(f"Tamanho após tratamento de nulos: {df_clean.shape}")
### Encontrando `continent`, `country` e `subnational` com Geopy
from geopy.geocoders import Nominatim
import pycountry_convert as pc
import time
from tqdm import tqdm

tqdm.pandas()

geolocator = Nominatim(user_agent="projeto_sismico_ufrj")

# A cache agora vai guardar um tuplo: (pais, subnacional)
cache_localizacao = {}


def obter_localizacao_por_coordenadas(lat, lon):
    coord_arredondada = (round(lat, 1), round(lon, 1))

    if coord_arredondada in cache_localizacao:
        return cache_localizacao[coord_arredondada]

    try:
        time.sleep(1.1)  # Respeitar o limite do servidor

        local = geolocator.reverse(f"{lat}, {lon}", language='en', exactly_one=True)

        if local and 'address' in local.raw:
            endereco = local.raw['address']
            pais = endereco.get('country', 'Ocean/Offshore')

            # A chave 'state' costuma guardar a informação subnacional
            # Caso o sismo seja num país pequeno ou no mar, pode não existir 'state'
            subnacional = endereco.get('state', 'Unknown')
        else:
            pais = 'Ocean/Offshore'
            subnacional = 'Unknown'

        resultado = (pais, subnacional)
        cache_localizacao[coord_arredondada] = resultado
        return resultado

    except Exception as e:
        return ('Unknown', 'Unknown')


import pandas as pd

print("A iniciar a busca geográfica por País e Estado (Isto pode demorar)...")

#df_exemplo = df_clean.head(100).copy()

## 1. Aplicar a função e dividir o resultado em duas colunas instantaneamente

# df_exemplo[['country', 'subnational']] = df_exemplo.progress_apply(
#     lambda row: pd.Series(obter_localizacao_por_coordenadas(row['latitude'], row['longitude'])),
#     axis=1
# )

df_clean[['country', 'subnational']] = df_clean.progress_apply(
    lambda row: pd.Series(obter_localizacao_por_coordenadas(row['latitude'], row['longitude'])),
    axis=1
)


# 2. Função para traduzir o País em Continente (Mantém-se igual)
def obter_continente(pais):
    if pais in ['Ocean/Offshore', 'Unknown']:
        return pais
    try:
        if pais == 'United States':
            pais = 'United States of America'
        elif pais == 'The Bahamas':
            pais = 'Bahamas'

        codigo_pais = pc.country_name_to_country_alpha2(pais, cn_name_format="default")
        codigo_continente = pc.country_alpha2_to_continent_code(codigo_pais)

        continentes_map = {
            'NA': 'North America', 'SA': 'South America', 'AS': 'Asia',
            'EU': 'Europe', 'AF': 'Africa', 'OC': 'Oceania', 'AN': 'Antarctica'
        }
        return continentes_map.get(codigo_continente, 'Unknown')
    except:
        return 'Unknown'


#print("\nA mapear os países para os respetivos continentes...")
#df_exemplo['continent'] = df_exemplo['country'].apply(obter_continente)

#print("\nConcluído! Visualização das novas colunas geográficas:")
#display(df_exemplo[['latitude', 'longitude', 'country', 'subnational', 'continent']])

print("\nA mapear os países para os respetivos continentes...")
df_clean['continent'] = df_clean['country'].apply(obter_continente)

print("\nConcluído! Visualização das novas colunas geográficas:")
display(df_clean[['latitude', 'longitude', 'country', 'subnational', 'continent']].head(100))
import pandas as pd

print("A preparar a comparação de países...")

# 1. Vamos assumir que a sua etapa de limpeza (df_clean) envolve remover
# os sismos que aconteceram no meio do oceano ou que não foram identificados.
# (Se a sua limpeza for diferente, basta ajustar esta linha)
#df_clean = df[~df['country'].isin(['Ocean/Offshore', 'Unknown'])].copy()

## 2. Agora SIM, a coluna 'country' existe no 'df' original e podemos contar!
#contagem_antes = df_exemplo['country'].value_counts(dropna=False)
#contagem_depois = df_exemplo['country'].value_counts()

# 2. Agora SIM, a coluna 'country' existe no 'df' original e podemos contar!
contagem_antes = df_clean['country'].value_counts(dropna=False)
contagem_depois = df_clean['country'].value_counts()

# 3. Criar o DataFrame de comparação lado a lado
df_comparacao = pd.DataFrame({
    'Antes da Limpeza (Total)': contagem_antes,
    'Depois da Limpeza (Válidos)': contagem_depois
})

# Substituir os valores nulos (NaN) por 0, caso algum país tenha desaparecido na limpeza,
# e converter para números inteiros para ficar bonito
df_comparacao = df_comparacao.fillna(0).astype(int)

# Ordenar para ver quem perdeu mais dados (opcional)
df_comparacao = df_comparacao.sort_values(by='Antes da Limpeza (Total)', ascending=False)

print("\n📊 Tabela de Comparação de Países (Top 15):")
display(df_comparacao)
### Linhas Duplicadas
# 1. Verificar quantas linhas estão exatamente duplicadas
quantidade_duplicados = df_clean.duplicated().sum()
print(f"Linhas duplicadas encontradas antes: {quantidade_duplicados}")
df_clean = df_clean.drop_duplicates()
quantidade_duplicados = df_clean.duplicated().sum()
print(f"Linhas duplicadas encontradas antes: {quantidade_duplicados}")
# Salvando o Dataset
# Visualizar o dataset higienizado
print(df_clean.head())

# Salvar o novo dataset limpo
arquivo_limpo = 'Dataset/alert_earthquakes_cleaned_with_null.csv'
df_clean.to_csv(arquivo_limpo, index=False)

print(f"\nHigienização concluída! O dataset limpo foi salvo como: {arquivo_limpo}")